# Reload the two BiLSTM snapshots

Personal eval notebook. Loads `model/best_model_{single,multi}_modal` and runs `evaluate` on the same preprocess as training.

**This checkout cannot actually reload those snapshots.** The folders have `saved_model.pb` + `keras_metadata.pb` and no `variables/` shard. The printed 0.8635 / 0.8735 numbers are from June 2023 when the weights were still on disk.

`X_val = X_test` is not a real validation split — I was staring at test while I trained. Pad length on the saved graphs is 78.

GloVe path here is `glove.twitter.27B.200d.bin`. Sentence paths have no `dataset/` prefix.


In [2]:
import tensorflow as tf
from gensim.models import KeyedVectors
from data_utils import Preprocess, preprocess_test, ReadOpen

In [3]:
glove_model = KeyedVectors.load_word2vec_format('glove.twitter.27B.200d.bin', binary=True)
emoji2vec_model = KeyedVectors.load_word2vec_format('emoji2vec_twitter.bin', binary=True)

In [4]:
filename_train = "train_sentence.csv"
Labelfile_train = "train_label.csv"
filename_test = "test_sentence.csv"
Labelfile_test = "test_label.csv"
filename_subtest = "subtest_sentence.csv"
Labelfile_subtest = "subtest_label.csv"
print('Reading data...')
data_train,labels_train,count_train = ReadOpen(filename_train,Labelfile_train)
data_test,labels_test,count_test = ReadOpen(filename_test,Labelfile_test)
data_subtest,labels_subtest,count_subtest = ReadOpen(filename_subtest,Labelfile_subtest)
print('Getting Embeddings...')
padded_docs_train, embedding_matrix,l,t = Preprocess(data_train,count_train,glove_model,emoji2vec_model)
padded_docs_test = preprocess_test(t,l,data_test)
padded_docs_subtest = preprocess_test(t,l,data_subtest)
print('Successfully loaded and processed the data!')


Reading data...
Getting Embeddings...
Successfully loaded and processed the data!


In [5]:
X_subtest = padded_docs_subtest
y_subtest = labels_subtest
X_train = padded_docs_train
X_test = padded_docs_test
y_train = labels_train
y_test = labels_test
# Not a validation split. I used test as the screen I watched while
# picking snapshots. Do not treat later acc as a locked holdout.
X_val = X_test
y_val = y_test

In [10]:
model_we = tf.keras.models.load_model("model/best_model_multi_modal")

In [11]:
loss, accuracy_we = model_we.evaluate(X_test, y_test, verbose=1)
loss, sub_accuracy_we = model_we.evaluate(X_subtest, y_subtest, verbose=1)
# loss1, accuracy1 = model.evaluate(X_test1, y_test1, verbose=1)

9/9 [==============================] - 0s 11ms/step - loss: 0.2852 - acc: 0.8921


In [19]:
model_we.summary()

Model: "sequential_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
module_wrapper_36 (ModuleWra (None, 78, 200)           0         
_________________________________________________________________
module_wrapper_37 (ModuleWra (None, 78, 200)           0         
_________________________________________________________________
bidirectional_12 (Bidirectio (None, 78, 512)           935936    
_________________________________________________________________
module_wrapper_38 (ModuleWra (None, 78, 512)           0         
_________________________________________________________________
bidirectional_13 (Bidirectio (None, 78, 512)           1574912   
_________________________________________________________________
module_wrapper_39 (ModuleWra (None, 78, 512)           0         
_________________________________________________________________
module_wrapper_40 (ModuleWra (None, 512)              

In [12]:
model_w = tf.keras.models.load_model("model/best_model_single_modal")

In [13]:
loss, accuracy_w = model_w.evaluate(X_test, y_test, verbose=1)
loss, sub_accuracy_w = model_w.evaluate(X_subtest, y_subtest, verbose=1)

9/9 [==============================] - 0s 11ms/step - loss: 0.3056 - acc: 0.8669


In [14]:
model_w.summary()

Model: "sequential_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
module_wrapper_30 (ModuleWra (None, 78, 200)           0         
_________________________________________________________________
module_wrapper_31 (ModuleWra (None, 78, 200)           0         
_________________________________________________________________
bidirectional_10 (Bidirectio (None, 78, 512)           935936    
_________________________________________________________________
module_wrapper_32 (ModuleWra (None, 78, 512)           0         
_________________________________________________________________
bidirectional_11 (Bidirectio (None, 78, 512)           1574912   
_________________________________________________________________
module_wrapper_33 (ModuleWra (None, 78, 512)           0         
_________________________________________________________________
module_wrapper_34 (ModuleWra (None, 512)              